In [566]:
import folium
from folium import plugins
import webbrowser
import pandas as pd
import numpy as np

In [567]:
#### Set filters ############################################
## Note: Map performance drops at around 400 groups
# Set to a number or NONE for no filter
MIN_GROUP_SIZE = 1000
DAY_TYPE = 0  # 0 = weekday, 1 = Saturday, 2 = Sunday
START_HOUR = 13 # Todo: this is still in UTC time, need to fix
###############################################################

# Load in data and filter
# Drop any row with NaN
data = pd.read_csv("../data/small_medium_merged.csv", sep=",").dropna()
if MIN_GROUP_SIZE is not None:
    data = data[(data["Count"]>= MIN_GROUP_SIZE)]

if DAY_TYPE is not None:
    data = data[data["day_type"] == DAY_TYPE]

if START_HOUR is not None:
    data = data[data["start_hour"] == START_HOUR]

# Define column names for convenience
pickup_lat = "Pickup Centroid Latitude"
pickup_long = "Pickup Centroid Longitude"
dropoff_lat = "Dropoff Centroid Latitude"
dropoff_long = "Dropoff Centroid Longitude"

# Adding columns for later reformatting:
# - pickup_point: (lat, long) tuple
# - dropoff_point: (lat, long) tuple
# - points: this is a "frozen set" of {pickup_point, dropoff_point}, it is an unordered
# but hashable type of set that we can use as to group sets of points
data["pickup_point"] = data.apply(lambda row: (row[pickup_lat],row[pickup_long]), axis = 1)
data["dropoff_point"] = data.apply(lambda row: (row[dropoff_lat],row[dropoff_long]), axis=1)
data["points"] = data.apply(lambda row: [row["pickup_point"], row["dropoff_point"]], axis=1).apply(frozenset)

# Create variable for transit time - rideshare time
# reformatting total time to int
data["totalTime"] = data["totalTime"].apply(lambda s: int(str(s)[:-1]))/60
data["Average Trip Minutes"] = data["Average Trip Seconds"]/60
data["trip_diff_time"] = data["totalTime"] - data["Average Trip Minutes"]

# Add colormapping for trip_diff_time
bins = [0, 0.25, 0.50, 0.75, 1]
labels = ["green", "yellow", "orange", "red"]
data["trip_diff_time_color"] = pd.cut(data["trip_diff_time"].rank(pct=True), bins=bins, labels=labels)


def standardize(x):
    """
    Standardizes value, used as weighting function for opacity/size
    """
    return (x-x.min())/(x.max()-x.min())

# data.sort_values("Count")
print("Ride Groups: ",data.shape[0])

Ride Groups:  270


In [568]:
# Create point sets data frame
# This is so that we can draw bidirectional routes
point_sets =pd.DataFrame(data.groupby(by=["points","pickup_point", "dropoff_point"])[["Count", "Average Trip Minutes","totalTime","trip_diff_time_color"]].first())
point_sets['line_weight'] = (standardize(point_sets['Count']) + 0.1) * 5
point_sets['opacity'] = standardize(point_sets['Count']) + 0.1
# point_sets.sort_values("Count")

In [569]:
# Create dataset of nodes
# Pickup and dropoff locations are often the same, so condense it so that
# we draw each node only once
pickup_nodes = data[[pickup_lat, pickup_long, "Count"]]
pickup_nodes.columns = ["latitude", "longitude", "Count"]
dropoff_nodes = data[[dropoff_lat, dropoff_long, "Count"]]
dropoff_nodes.columns = ["latitude", "longitude", "Count"]
all_nodes = pd.concat([pickup_nodes, dropoff_nodes], axis = 0)

# Radius and opacity are based on how many rides start OR end at that spot
all_nodes = all_nodes.groupby(["latitude", "longitude"])["Count"].agg(sum).reset_index()
all_nodes["radius"] = (standardize(all_nodes["Count"]) + 0.1) * 12
all_nodes["opacity"] = (standardize(all_nodes["Count"]) + 0.1) 
# all_nodes.sort_values(by = "group_n")


In [570]:
# Make the Map
map = folium.Map(
    location=(41.86721, -87.63231),
    zoom_control=False,
    tiles="Cartodb Positron", 
    zoom_start=13
)

# Plot map nodes
for i, node in all_nodes.iterrows():
    folium.CircleMarker(
        location=[node["latitude"], node["longitude"]],
        radius=node["radius"],
        color="beige",
        stroke=False,
        fill=True,
        fill_opacity=node["opacity"],
        tooltip= f"Node: {int(node["Count"])} rides",
        opacity=1).add_to(map)

# Plot routes
for index in point_sets.index.get_level_values(0):
    subset = point_sets[point_sets.index.get_level_values(0).isin([index])]
    for index, value in subset.iterrows():
        route = [index[1], index[2]]
        
        line = folium.plugins.PolyLineOffset(route, 
                                             color=value["trip_diff_time_color"], 
                                             opacity=value["opacity"],
                                             weight = value["line_weight"], 
                                             offset=-5,
                                             tooltip= f"Route: {int(value["Count"])} rides<br> \
                                                Transit Time: {round(value["totalTime"])}min<br>\
                                                Rideshare Time: {round(value["Average Trip Minutes"])}min").add_to(map)
        # Add arrows
        folium.plugins.PolyLineTextPath(line,
                                        text="\u2192",
                                        repeat=True,
                                        center = True,
                                        offset = 9,
                                        attributes={"font-size": "30", 
                                                    "fill" : value["trip_diff_time_color"],
                                                    "fill-opacity": str(value["opacity"])}).add_to(map)

    
map.save("map.html")
webbrowser.open("map.html")
map.show_in_browser()

Your map should have been opened in your browser automatically.
Press ctrl+c to return.
